<a href="https://colab.research.google.com/github/Isabela-Tellez/BootcampIA/blob/main/07.%20Julio-02/Proyecto_ML_Prediccion_Churn_Telco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Mi Guía de Machine Learning: Predicción de Churn (Telco)
### Módulo 3: Árboles de Decisión y Ensembles

---

**🎯 Mi Mapa de Ruta Técnico:**

| Paso | Lo que voy a hacer | ¿Por qué es crítico? |
| :--- | :--- | :--- |
| **1. Limpieza y Encoding** | Pasar texto a números (`pd.get_dummies`) | Scikit-Learn no entiende variables categóricas o strings. |
| **2. División Train/Test** | Separar datos en 80% y 20% | Entrenar con una parte y validar con otra para medir aprendizaje real. |
| **3. Árbol Simple** | Probar diferentes profundidades (`max_depth`) | Controlar el overfitting/underfitting y encontrar el balance. |
| **4. Ensemble** | Entrenar un Random Forest | Promediar árboles mediante *bagging* para estabilizar predicciones. |
| **5. Evaluación Ética** | Mirar la matriz de confusión y el *Recall* | El dataset está desbalanceado; el *Accuracy* me puede engañar. |

---

### 🗂️ Recordatorio del Dataset
Estoy usando el dataset **Telco Customer Churn (IBM)**. Contiene información sobre clientes de telecomunicaciones (antigüedad, contrato, gastos mensuales) y mi objetivo es predecir la columna objetivo `Churn` (si se van o se quedan).

---
## ⚙️ Setup

Stack de herramientas necesarias para todo el proyecto.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ Mi entorno de trabajo está listo.")

✅ Mi entorno de trabajo está listo.


## 📥 Paso 1: Carga de Datos
Cargo el archivo descargado para inspeccionar su estructura inicial y verificar los tipos de columnas.
> 💡 *Nota mental:* Si el archivo está en formato Excel, usar `pd.read_excel()`. Si es CSV, cambiar a `pd.read_csv()`.

In [6]:
df_raw = pd.read_excel("Telco_customer_churn.xlsx")
df_raw.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 🔍 Paso 2: Limpieza y Encoding Autónomo
Aquí transformo el dataset para dejarlo apto para los modelos.
* *Obligatorio:* Detectar columnas numéricas leídas como texto (ej. espacios en blanco en `TotalCharges`), forzar su conversión con `pd.to_numeric(..., errors='coerce')` y limpiar nulos.
* *Encoding:* Aplicar `pd.get_dummies(..., drop_first=True)` para transformar categorías cualitativas a binarias (`0` o `1`).

In [16]:
# Forzar conversión a número (TotalCharges)
df_raw['Total Charges'] = pd.to_numeric(df_raw['Total Charges'], errors='coerce')

#Limpieza de nulos
df_raw.dropna(subset=['Total Charges'], inplace=True)

# Eliminano identificadores y redundancias antes del Encoding
df_limpio = df_raw.drop(columns=['CustomerID', 'Lat Long', 'Churn Label', 'Churn Reason'])

# Codificar el resto de texto a números
df_encoded = pd.get_dummies(df_limpio, drop_first=True)

## ⚔️ Paso 3: División en Entrenamiento y Prueba
Defino mis matrices de características (`X`) y mi etiqueta objetivo (`y`). Aplico la división 80/20 fijando el `random_state=42` para asegurar que mis pruebas siempre sean reproducibles.

In [13]:
# Aislar la variable objetivo
# Separar la variable objetivo (y) de las predictoras (X)
y = df_encoded['Churn Value']
X = df_encoded.drop(columns=['Churn Value'])

# 2. Hacer la división 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Comprobar que las dimensiones sean correctas
print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

Datos de entrenamiento: 5625 filas
Datos de prueba: 1407 filas


## 🌳 Paso 4: Árbol de Decisión y Control de Profundidad
Voy a entrenar al menos dos árboles con profundidades distintas para evaluar el sobreajuste.
* **Árbol libre:** Sin límite de `max_depth`. Memorizará el set de entrenamiento pero fallará en validación (Overfitting).
* **Árbol controlado:** Poniendo un límite (ej. `max_depth=5`) para forzar al modelo a generalizar mejor.

In [14]:
# Entrenar y evaluar el Modelo 1 (Sin límites)
arbol_libre = DecisionTreeClassifier(random_state=42)
arbol_libre.fit(X_train, y_train)
preds_libre = arbol_libre.predict(X_test)

print("=== REPORTE: ÁRBOL SIN RESTRICCIONES ===")
print(classification_report(y_test, preds_libre))

# Entrenar y evaluar el Modelo 2 (Controlado)
arbol_controlado = DecisionTreeClassifier(max_depth=5, random_state=42)
arbol_controlado.fit(X_train, y_train)
preds_controlado = arbol_controlado.predict(X_test)

print("\n=== REPORTE: ÁRBOL CONTROLADO (max_depth=5) ===")
print(classification_report(y_test, preds_controlado))

=== REPORTE: ÁRBOL SIN RESTRICCIONES ===
              precision    recall  f1-score   support

           0       0.93      0.94      0.94      1012
           1       0.84      0.82      0.83       395

    accuracy                           0.91      1407
   macro avg       0.89      0.88      0.88      1407
weighted avg       0.91      0.91      0.91      1407


=== REPORTE: ÁRBOL CONTROLADO (max_depth=5) ===
              precision    recall  f1-score   support

           0       0.93      0.96      0.94      1012
           1       0.88      0.81      0.85       395

    accuracy                           0.92      1407
   macro avg       0.91      0.89      0.90      1407
weighted avg       0.92      0.92      0.92      1407



## 🌲 Paso 5: Ensamble con Random Forest
Entreno un `RandomForestClassifier`. Mi objetivo aquí es comprobar cómo la combinación de múltiples árboles mediante muestreo aleatorio (*bagging*) reduce la varianza y mejora la estabilidad frente a un árbol único.

In [17]:
# Instanciar y entrenar el modelo ensemble
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Generar las predicciones sobre el conjunto de test
predicciones_rf = rf_model.predict(X_test)

# Evaluar el rendimiento del bosque
print("=== REPORTE: RANDOM FOREST ===")
print(classification_report(y_test, predicciones_rf))

=== REPORTE: RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1012
           1       1.00      0.06      0.12       395

    accuracy                           0.74      1407
   macro avg       0.87      0.53      0.48      1407
weighted avg       0.81      0.74      0.64      1407



## ⚖️ Paso 6: Cuidado con el Desbalance (Métricas de Negocio)
Anoto mis conclusiones sobre los riesgos éticos y operativos del desbalance de clases en este problema:

1. **La trampa del Accuracy:** Como la gran mayoría de los clientes decide quedarse (clase minoritaria = abandono), una métrica de precisión general alta puede ocultar un modelo que es incapaz de detectar los abandonos reales.
2. **Consecuencias Financieras (Falsos Negativos):** Si mi modelo predice falsamente que un cliente **no** se va a ir, la empresa no tomará ninguna acción de retención. El cliente abandonará la compañía y perderemos su flujo de ingresos recurrente, haciendo que el costo operativo de no predecir a tiempo sea muy alto. El **Recall** de la clase `Churn` es mi métrica clave aquí.

---
## ✅ Checklist de mi Proceso

- [ ] Cargué correctamente el dataset original.
- [ ] Procesé los tipos numéricos erróneos y convertí las variables con One-Hot Encoding.
- [ ] Separé los conjuntos con una proporción del 20% para testeo.
- [ ] Evalué y comparé el efecto de modificar el hiperparámetro `max_depth`.
- [ ] Implementé el modelo ensemble Random Forest y analicé sus diferencias frente al árbol individual.
- [ ] Documenté el impacto operativo que tiene ignorar el desbalance de clases en la métrica final.